# Black Friday Sales Prediction

End-to-end analysis of the Kaggle Black Friday dataset.

**Goal:** understand customer purchase behaviour and build a multivariable linear regression model to predict `Purchase`.

The notebook uses the original Kaggle `train.csv` and `test.csv` files and does not fabricate data.

## 1. Setup

The notebook uses pandas and NumPy for data preparation, matplotlib for visualisation, and scikit-learn for regression and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
DATA_DIR = Path('../data')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Place the original Kaggle train.csv and test.csv files in projects/black-friday-sales-prediction/data/')

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print('Train shape:', train.shape)
print('Test shape:', test.shape)

## 2. Initial inspection

The labelled training data contains `Purchase`; the Kaggle test data does not. Both datasets receive the same cleaning and feature-engineering logic.

In [ ]:
print('Training columns:')
print(train.columns.tolist())
print('\nMissing values in training:')
display(train.isna().sum())
print('\nMissing values in test:')
display(test.isna().sum())

## 3. Basic exploratory analysis

These summaries provide context before modelling.

In [ ]:
print('Purchase summary:')
display(train['Purchase'].describe())
city_summary = train.groupby('City_Category')['Purchase'].agg(['count', 'mean']).round(2)
display(city_summary)
train.groupby('Age')['Purchase'].mean().sort_index().plot(kind='bar', figsize=(8, 4))
plt.title('Average Purchase by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Purchase')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Cleaning and feature engineering

- `User_ID` and `Product_ID` are identifiers and are excluded from the predictive feature set.
- Missing `Product_Category_2` and `Product_Category_3` values are represented through `Num_Categories`.
- `Age` and `Stay_In_Current_City_Years` are converted to ordinal values.
- Nominal categories are one-hot encoded.
- The same transformation logic is applied to train and test.

In [ ]:
AGE_MAP = {'0-17': 0, '18-25': 1, '26-35': 2, '36-45': 3, '46-50': 4, '51-55': 5, '55+': 6}
STAY_MAP = {'0': 0, '1': 1, '2': 2, '3': 3, '4+': 4}

def prepare_features(df):
    out = df.drop(columns=['User_ID', 'Product_ID'], errors='ignore').copy()
    out['Num_Categories'] = 1 + out['Product_Category_2'].notna().astype(int) + out['Product_Category_3'].notna().astype(int)
    out['Gender_M'] = (out['Gender'] == 'M').astype(int)
    out['Age_ord'] = out['Age'].map(AGE_MAP)
    out['Stay_ord'] = out['Stay_In_Current_City_Years'].astype(str).map(STAY_MAP)
    occ = pd.get_dummies(out['Occupation'], prefix='Occ', drop_first=True, dtype=int)
    city = pd.get_dummies(out['City_Category'], prefix='City', drop_first=True, dtype=int)
    pc1 = pd.get_dummies(out['Product_Category_1'], prefix='PC1', drop_first=True, dtype=int)
    base = out[['Gender_M', 'Age_ord', 'Stay_ord', 'Marital_Status', 'Num_Categories']]
    return pd.concat([base, occ, city, pc1], axis=1).astype(float)

X = prepare_features(train)
X_test_kaggle = prepare_features(test)
X_test_kaggle = X_test_kaggle.reindex(columns=X.columns, fill_value=0)
y = train['Purchase'].astype(float)
print('Model feature matrix:', X.shape)
print('Kaggle test feature matrix:', X_test_kaggle.shape)

## 5. Model training

An 80/20 hold-out split is used to evaluate the regression model on labelled observations it did not train on.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.20, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))
print('Intercept:', model.intercept_)

## 6. Evaluate the regression model

R² measures explained variance. MAE and RMSE express prediction error in the same units as `Purchase`; RMSE penalises larger errors more heavily.

In [ ]:
y_pred = model.predict(X_valid)
r2 = r2_score(y_valid, y_pred)
n, p = X_valid.shape
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae = mean_absolute_error(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)
metrics = pd.Series({'R2': r2, 'Adjusted R2': adjusted_r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse})
display(metrics.to_frame('Value'))

## 7. Actual vs predicted

A diagnostic comparison of observed purchase values with model predictions.

In [ ]:
comparison = pd.DataFrame({'Actual': y_valid, 'Predicted': y_pred})
plt.figure(figsize=(7, 7))
plt.scatter(comparison['Actual'], comparison['Predicted'], alpha=0.15)
limits = [min(comparison['Actual'].min(), comparison['Predicted'].min()), max(comparison['Actual'].max(), comparison['Predicted'].max())]
plt.plot(limits, limits, linestyle='--')
plt.title('Actual vs Predicted Purchase')
plt.xlabel('Actual Purchase')
plt.ylabel('Predicted Purchase')
plt.tight_layout()
plt.show()

## 8. Coefficient interpretation

One-hot encoded coefficients are interpreted relative to their omitted reference category.

In [ ]:
coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_}).sort_values('Coefficient', ascending=False)
display(coefficients.head(10))
display(coefficients.tail(10))

## 9. Statistical significance

`LinearRegression` provides coefficients but not p-values. An equivalent OLS model is used as a supplementary statistical diagnostic.

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols = sm.OLS(y_train, X_train_sm).fit()
significance = pd.DataFrame({'Coefficient': ols.params, 'Std_Error': ols.bse, 'p_value': ols.pvalues}).sort_values('p_value')
display(significance.head(15))

## 10. Generate predictions for Kaggle `test.csv`

The Kaggle test file has no `Purchase` target, so its predictions cannot be evaluated against ground truth here. The trained model can generate a prediction for every test record.

In [ ]:
test_predictions = model.predict(X_test_kaggle)
submission = test.copy()
submission['Predicted_Purchase'] = test_predictions
display(submission[['User_ID', 'Product_ID', 'Predicted_Purchase']].head())

## 11. Save predictions locally

The generated file is ignored by Git and therefore is not committed to the public source repository.

In [ ]:
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)
prediction_path = output_dir / 'black_friday_test_predictions.csv'
submission.to_csv(prediction_path, index=False)
print(f'Saved predictions to: {prediction_path}')

## 12. Conclusion

This baseline demonstrates a complete regression workflow: inspect the raw data, apply consistent cleaning, engineer useful features, encode categorical variables, train a multivariable linear regression model, evaluate it on held-out labelled data, interpret coefficients, and generate predictions for the original Kaggle test set.

Future iterations can compare regularised regression or tree-based models and add cross-validation and residual diagnostics.